# Path A, P13 — sensitivities (Oct 28, 2026)

Pre-registered in the Path A Workbook, Ch. A8.6. No choice below was changed. G-val is reused for robustness only. No Ring or CT. P10–P12 outputs are read, not refit or overwritten.

**Base setup = P11:** R0 structure (const + lab + features), all training cells (Hierarchy, Full, Star), fit on G-train, outcome `y_rev`, `fit.fit` (statsmodels Logit, unpenalized, raw features). M1 = A, |s| (`abs_s`), `at_end`; M2 = M1 + trend; M2' = M1 + e, r (`time_at_end_log1p`, `time_since_rev_log1p`). M3 is not fit.

**New tables** (P8 run set `load_runs("behavior", roles={"train", "val"})` and `hazard.build_hazard_table`; git-ignored parquets in `data/processed/`): S1 b = 0.1 s; S2 b = 0.5 s; S3 m = 3 (b = 0.25 s). Per cell: rows, dropped mid-event bins, sum(y_rev), sum(y_probe); P8's leak counters (bot-free complete subset, as in P8); missing values.

**Sensitivities** (each changes one thing; S4–S8 use the primary table `hazard_table.parquet`):
- S1–S3: the rebuilt tables
- S4: abs_s dropped from M1, M2, M2'
- S5: e, r replaced by the capped columns min(t, 10 s): `time_at_end_cap10`, `time_since_rev_cap10`
- S6: bot-run rows dropped (train and val)
- S7: old-CL interface rows (`interface == "old_cl"`) dropped (train and val)
- S8 (descriptive): A replaced by a natural cubic spline (`fit.rcs`, Harrell's form: A + 2 nonlinear columns), 4 knots at G-train quantiles 0.05 / 0.35 / 0.65 / 0.95 of A; same knots and formula on G-val

**Reported per sensitivity:** checks 1 (converged, betas finite, no warnings) and 2 (sum(h_hat) == sum(y_rev): overall, lab, cloud) for M1, M2, M2'; M2' weights with SEs next to the primary M2' weights and the primary bootstrap 95% CI (`p12_beta_ci.csv`), flags sign same? / inside CI? (no CI flag for const, at_end in S4, e / r in S5, A in S8); S1, S2: const shift vs ln(b / 0.25); S5: hazard factor at 10 s, capped exp(10 beta) vs primary exp(beta ln 11); S8: hazard factor at A = 0.25 / 0.5 / 0.75 / 1 vs A = 0, spline vs linear (primary beta_A). Held-out: M2 − M1 and M2' − M1 delta per G-val bin with 95% CI (`fit.boot_delta`, 10,000 G-val group resamples, seed 573, betas fixed); S1–S3 on their own tables' G-val bins. Flags: recency CI above 0? (M2' − M1) trend CI includes 0? (M2 − M1).

Saved: `results/tables/p13_tables_check.csv`, `p13_weights.csv`, `p13_verdicts.csv`.

In [1]:
import warnings

import numpy as np
import pandas as pd

from slider import config
from slider import fit
from slider import hazard
from slider import slider_logs as sl
from slider.labclock import CELLS, lab_log
from slider.reversals import run_events, to_steps

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

## New tables S1–S3

In [2]:
LEAKS = ["leak: row at time >= t", "leak: abs_s not from t_row", "leak: anchor on an unresolved reversal", "anchor not found"]


def leak_counts(h, b, m):
    """P8's leak counters (p8_hazard_table.ipynb) for a table built with (b, m), on the bot-free complete subset."""
    sub = h[~h["bot_run"] & ~h["incomplete"]]
    kept = {key: g for key, g in sub.groupby(["round_id", "player"])}
    rows = []
    for run in runs:
        if run["bot"] or run["end"] == "incomplete":
            continue
        if run["dataset"] == "CC":
            t_lab, X_lab = lab_log(run["t"], run["X"])
            run = {**run, "t": t_lab, "X": X_lab}
        s = sl.sign_correct(run)
        k = to_steps(s)
        ev = run_events(run, m)
        rev = ev[ev["type"] == "reversal"]
        for j, bins in hazard.at_risk_samples(run, b).groupby("player"):
            times = bins["time"].to_numpy()
            g = kept.get((f"{run['file']}#w{run['wave']}", j), sub.iloc[:0])
            t_kept, anchors = g["t"].to_numpy(), (g["t"] - g["time_since_rev"]).to_numpy()
            r = rev[rev["player"] == j]
            first = r["time"].to_numpy()
            done = np.array([run["t"][x.event_row + int(np.argmax(np.cumsum(np.abs(np.diff(k[:, j]))[x.event_row - 1:]) >= m))]
                             for x in r.itertuples()], dtype=float)                 # the row where it reaches m steps
            hit = np.abs(anchors[:, None] - first[None, :]) < 1e-6
            s_row = np.abs(s[np.searchsorted(run["t"], g["t_row"].to_numpy() - 1e-9), j])
            rows.append({"topology": sl.label_topology(run["A"]), "dataset": run["dataset"],
                         LEAKS[0]: int(np.sum(g["t_row"].to_numpy() >= t_kept - 1e-9)),
                         LEAKS[1]: int(np.sum(np.abs(s_row - g["abs_s"].to_numpy()) > 1e-9)),
                         LEAKS[2]: int(np.sum(hit & (done[None, :] >= t_kept[:, None] - 1e-6))),
                         LEAKS[3]: int(np.sum(~hit.any(axis=1) & (np.abs(anchors - times[0]) > 1e-6)))})
    return pd.DataFrame(rows).groupby(["topology", "dataset"]).sum()


def per_cell(h, d):
    keys = ["topology", "dataset"]
    g = h.groupby(keys).agg(rows=("y_rev", "size"), sum_y_rev=("y_rev", "sum"), sum_y_probe=("y_probe", "sum"))
    g.insert(1, "dropped", d.groupby(keys).size().reindex(g.index, fill_value=0))
    g["missing_table"] = h.isna().sum(axis=1).groupby([h["topology"], h["dataset"]]).sum()
    g["missing_dropped"] = d.isna().sum(axis=1).groupby([d["topology"], d["dataset"]]).sum().reindex(g.index, fill_value=0)
    return g


runs, table = sl.load_runs("behavior", roles={"train", "val"})      # the P8 run set
SETTINGS = {"S1": (0.1, 2), "S2": (0.5, 2), "S3": (0.25, 3)}         # (b, m)
PATHS = {"S1": config.HAZARD_TABLE_S1, "S2": config.HAZARD_TABLE_S2, "S3": config.HAZARD_TABLE_S3}
parts = []
for name, (b, m) in SETTINGS.items():                               # one table at a time (memory)
    h, d = hazard.build_hazard_table(runs, b=b, m=m)
    h.to_parquet(PATHS[name], index=False)
    print(name, f"b = {b}, m = {m}:", len(h), "rows,", len(d), "dropped,", h["round_id"].nunique(), "rounds,",
          h["group_id"].nunique(), "groups")
    c = per_cell(h, d).join(leak_counts(h, b, m)).reindex(CELLS)
    parts.append(c.reset_index().assign(table=name, b=b, m=m))
    del h, d
del runs
tables_check = pd.concat(parts, ignore_index=True)
tables_check = tables_check[["table", "b", "m", "topology", "dataset", "rows", "dropped", "sum_y_rev", "sum_y_probe"]
                            + LEAKS + ["missing_table", "missing_dropped"]]
tables_check.to_csv(config.P13_TABLES_CHECK, index=False)
print("all leak counters 0:", bool((tables_check[LEAKS] == 0).all().all()),
      "| no missing values:", bool((tables_check[["missing_table", "missing_dropped"]] == 0).all().all()))
tables_check

S1 b = 0.1, m = 2: 1400967 rows, 87977 dropped, 365 rounds, 60 groups


S2 b = 0.5, m = 2: 281223 rows, 17536 dropped, 365 rounds, 60 groups


S3 b = 0.25, m = 3: 547627 rows, 48697 dropped, 365 rounds, 60 groups


all leak counters 0: True | no missing values: True


,table,b,m,topology,dataset,rows,dropped,sum_y_rev,sum_y_probe,leak: row at time >= t,leak: abs_s not from t_row,leak: anchor on an unresolved reversal,anchor not found,missing_table,missing_dropped
0,S1,0.10,2,Hierarchy,EL,240580,12613,1610,381,0,0,0,0,0,0
1,S1,0.10,2,Hierarchy,CL,218278,13337,1229,299,0,0,0,0,0,0
2,S1,0.10,2,Hierarchy,CC,210893,9119,1417,406,0,0,0,0,0,0
3,S1,0.10,2,Full,CL,393858,32137,2863,716,0,0,0,0,0,0
4,S1,0.10,2,Full,CC,197770,12691,1916,370,0,0,0,0,0,0
5,S1,0.10,2,Star,CC,139588,8080,938,230,0,0,0,0,0,0
6,S2,0.50,2,Hierarchy,EL,48286,2500,1585,366,0,0,0,0,0,0
7,S2,0.50,2,Hierarchy,CL,43792,2675,1209,275,0,0,0,0,0,0
8,S2,0.50,2,Hierarchy,CC,42378,1804,1390,374,0,0,0,0,0,0
9,S2,0.50,2,Full,CL,78913,6436,2828,651,0,0,0,0,0,0


## Sensitivities S1–S8: fits on G-train, checks 1 and 2

In [3]:
M1 = ["A", "abs_s", "at_end"]
ER = ["time_at_end_log1p", "time_since_rev_log1p"]
CAP = ["time_at_end_cap10", "time_since_rev_cap10"]


def models(m1, er):
    return {"M1": m1, "M2": m1 + ["trend"], "M2'": m1 + er}


COLS = ["split", "topology", "dataset", "group_id", "interface", "bot_run", "y_rev", "A", "trend", "abs_s", "at_end"] + ER + CAP
primary = pd.read_parquet(config.HAZARD_TABLE, columns=COLS)
primary = primary[primary["topology"].isin(config.TRAIN_TOPOLOGIES)]

KNOTS = np.quantile(primary.loc[primary["split"] == "train", "A"], [0.05, 0.35, 0.65, 0.95])     # S8: on G-train
basis = fit.rcs(primary["A"], KNOTS)                                                           # same basis on G-val
spline = primary.assign(A_rcs1=basis[:, 0], A_rcs2=basis[:, 1])
print("S8 knots (G-train quantiles of A):", KNOTS)


def data(name):
    """S1-S3: the rebuilt table; S4-S8: the primary table (S6, S7 filtered; S8 with the spline columns)."""
    if name in PATHS:
        return pd.read_parquet(PATHS[name], columns=COLS)
    if name == "S6":
        return primary[~primary["bot_run"]]
    if name == "S7":
        return primary[primary["interface"] != "old_cl"]
    return spline if name == "S8" else primary


MODELS = {"S1": models(M1, ER), "S2": models(M1, ER), "S3": models(M1, ER), "S4": models(["A", "at_end"], ER),
          "S5": models(M1, CAP), "S6": models(M1, ER), "S7": models(M1, ER),
          "S8": models(["A", "A_rcs1", "A_rcs2", "abs_s", "at_end"], ER)}
LABEL = {"S8": "descriptive"}


def flags(lo_trend, hi_trend, lo_rec):
    return {"recency_ci_above_0": bool(lo_rec > 0), "trend_ci_includes_0": bool(lo_trend <= 0 <= hi_trend)}


p11 = pd.read_csv(config.P11_2X2, float_precision="round_trip").set_index("comparison")
t, r = p11.loc["M2 - M1"], p11.loc["M2' - M1"]
verdicts = [{"sensitivity": "primary (P11)", "n_train_bins": np.nan, "n_val_bins": t["n_val_bins"], "n_val_groups": t["n_val_groups"],
             "delta_M2_M1": t["delta"], "ci_low_M2_M1": t["ci_low"], "ci_high_M2_M1": t["ci_high"],
             "delta_M2p_M1": r["delta"], "ci_low_M2p_M1": r["ci_low"], "ci_high_M2p_M1": r["ci_high"],
             **flags(t["ci_low"], t["ci_high"], r["ci_low"])}]
params, bse_m2p, checks = {}, {}, []
for name, mods in MODELS.items():
    h = data(name)
    h = h[h["topology"].isin(config.TRAIN_TOPOLOGIES)]
    train, val = h[h["split"] == "train"], h[h["split"] == "val"]
    lab = train["dataset"].isin(["EL", "CL"]).to_numpy()
    y = train["y_rev"].to_numpy()
    fitted = {}
    for mname, feats in mods.items():
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mod = fit.fit(train, feats, False)
        fitted[mname] = mod
        ok = bool(mod.mle_retvals["converged"] and np.isfinite(mod.params).all() and not caught)
        checks.append({"sensitivity": name, "model": mname, "check": 1, "what": "converged, betas finite, no warnings",
                       "numbers": f"converged={mod.mle_retvals['converged']}, iterations={mod.mle_retvals['iterations']}, "
                                  f"finite={np.isfinite(mod.params).all()}, warnings={len(caught)}"
                                  + "".join(f" | {x.category.__name__}: {x.message}" for x in caught), "pass": ok})
        hh = np.asarray(mod.predict())
        for part, mask in [("overall", np.ones(len(train), bool)), ("lab", lab), ("cloud", ~lab)]:
            diff = hh[mask].sum() - y[mask].sum()
            checks.append({"sensitivity": name, "model": mname, "check": 2, "what": f"sum(h_hat) == sum(y_rev), {part}",
                           "numbers": f"sum h = {hh[mask].sum():.6f}, sum y = {y[mask].sum()}, diff = {diff:.2e}",
                           "pass": bool(abs(diff) < 1e-3)})
    # held-out: M2 - M1 and M2' - M1 on this sensitivity's G-val bins, betas fixed
    ll = {m: fit.bin_ll(fitted[m], val) for m in fitted}
    dt, (lt, ht_) = fit.boot_delta(ll["M2"], ll["M1"], val["group_id"])
    dr, (lr, hr) = fit.boot_delta(ll["M2'"], ll["M1"], val["group_id"])
    verdicts.append({"sensitivity": name, "n_train_bins": len(train), "n_val_bins": len(val), "n_val_groups": val["group_id"].nunique(),
                     "delta_M2_M1": dt, "ci_low_M2_M1": lt, "ci_high_M2_M1": ht_,
                     "delta_M2p_M1": dr, "ci_low_M2p_M1": lr, "ci_high_M2p_M1": hr, **flags(lt, ht_, lr)})
    params[name] = {m: fitted[m].params for m in fitted}
    bse_m2p[name] = fitted["M2'"].bse
    del h, train, val, fitted, ll
checks = pd.DataFrame(checks)
print("all checks 1 and 2 pass:", bool(checks["pass"].all()))
with pd.option_context("display.max_colwidth", 200):
    display(checks)

S8 knots (G-train quantiles of A): [0.05       0.48571429 0.85       1.        ]


all checks 1 and 2 pass: True


,sensitivity,model,check,what,numbers,pass
0,S1,M1,1,"converged, betas finite, no warnings","converged=True, iterations=10, finite=True, warnings=0",True
1,S1,M1,2,"sum(h_hat) == sum(y_rev), overall","sum h = 6099.000000, sum y = 6099, diff = 8.19e-12",True
2,S1,M1,2,"sum(h_hat) == sum(y_rev), lab","sum h = 3256.000000, sum y = 3256, diff = 4.55e-12",True
3,S1,M1,2,"sum(h_hat) == sum(y_rev), cloud","sum h = 2843.000000, sum y = 2843, diff = 3.64e-12",True
4,S1,M2,1,"converged, betas finite, no warnings","converged=True, iterations=10, finite=True, warnings=0",True
5,S1,M2,2,"sum(h_hat) == sum(y_rev), overall","sum h = 6099.000000, sum y = 6099, diff = 3.64e-12",True
6,S1,M2,2,"sum(h_hat) == sum(y_rev), lab","sum h = 3256.000000, sum y = 3256, diff = 3.18e-12",True
7,S1,M2,2,"sum(h_hat) == sum(y_rev), cloud","sum h = 2843.000000, sum y = 2843, diff = -1.36e-12",True
8,S1,M2',1,"converged, betas finite, no warnings","converged=True, iterations=10, finite=True, warnings=0",True
9,S1,M2',2,"sum(h_hat) == sum(y_rev), overall","sum h = 6099.000000, sum y = 6099, diff = -9.09e-13",True


## Held-out: M2 − M1 and M2' − M1 on G-val, per bin

In [4]:
verdicts = pd.DataFrame(verdicts)
verdicts["checks_1_2_pass"] = verdicts["sensitivity"].map(checks.groupby("sensitivity")["pass"].all())
verdicts["label"] = verdicts["sensitivity"].map(LABEL).fillna("")
verdicts.to_csv(config.P13_VERDICTS, index=False)
with pd.option_context("display.float_format", "{:.3e}".format):
    display(verdicts)

,sensitivity,n_train_bins,n_val_bins,n_val_groups,delta_M2_M1,ci_low_M2_M1,ci_high_M2_M1,delta_M2p_M1,ci_low_M2p_M1,ci_high_M2p_M1,recency_ci_above_0,trend_ci_includes_0,checks_1_2_pass,label
0,primary (P11),NaN,225033,21,2.091e-04,-1.280e-04,4.743e-04,2.862e-03,1.531e-03,4.748e-03,True,True,NaN,
1,S1,8.390e+05,561988,21,6.303e-05,-4.749e-05,1.506e-04,1.071e-03,5.626e-04,1.783e-03,True,True,True,
2,S2,1.685e+05,112759,21,5.711e-04,-2.192e-04,1.217e-03,6.065e-03,3.271e-03,1.004e-02,True,True,True,
3,S3,3.280e+05,219586,21,1.360e-04,-1.810e-04,3.776e-04,2.661e-03,1.306e-03,4.632e-03,True,True,True,
4,S4,3.361e+05,225033,21,2.198e-04,-1.242e-04,4.915e-04,2.846e-03,1.517e-03,4.723e-03,True,True,True,
5,S5,3.361e+05,225033,21,2.091e-04,-1.280e-04,4.743e-04,2.732e-03,1.400e-03,4.689e-03,True,True,True,
6,S6,2.989e+05,185558,16,2.030e-04,-2.644e-04,5.375e-04,2.843e-03,1.220e-03,5.226e-03,True,True,True,
7,S7,3.016e+05,175061,18,1.739e-04,-2.095e-04,4.700e-04,3.304e-03,1.673e-03,5.698e-03,True,True,True,
8,S8,3.361e+05,225033,21,2.476e-04,-1.634e-04,5.534e-04,2.784e-03,1.465e-03,4.695e-03,True,True,True,descriptive


## M2' weights vs the primary M2' (P12 estimate and bootstrap 95% CI), and derived quantities

In [5]:
p12 = pd.read_csv(config.P12_BETA_CI, float_precision="round_trip").set_index("quantity")
p11c = pd.read_csv(config.P11_COEFS, float_precision="round_trip").set_index(["model", "term"])
REF = {"time_at_end_cap10": "time_at_end_log1p", "time_since_rev_cap10": "time_since_rev_log1p"}
NO_CI = {"S4": {"at_end"}, "S5": set(CAP), "S8": {"A"}}

rows = []
for name in MODELS:
    for term, beta in params[name]["M2'"].items():
        ref = REF.get(term, term)
        row = {"sensitivity": name, "kind": "M2' weight", "name": term, "value": beta, "se": bse_m2p[name][term]}
        if ref in p12.index:
            p = p12.loc[ref]
            row.update(reference_term=ref, reference=p["estimate"], ref_ci_low=p["ci_low"], ref_ci_high=p["ci_high"],
                       sign_same=bool(np.sign(beta) == np.sign(p["estimate"])))
            if term != "const" and term not in NO_CI.get(name, set()):
                row["inside_ci"] = bool(p["ci_low"] <= beta <= p["ci_high"])
        rows.append(row)

for name in ["S1", "S2"]:                       # const shift vs ln(b / 0.25)
    b = SETTINGS[name][0]
    for mname in ["M1", "M2", "M2'"]:
        rows.append({"sensitivity": name, "kind": "const shift vs ln(b/0.25)", "name": mname,
                     "value": params[name][mname]["const"] - p11c.loc[(mname, "const"), "beta"], "reference": np.log(b / 0.25)})

for term in CAP:                                # S5: hazard factor at 10 s, capped vs primary
    rows.append({"sensitivity": "S5", "kind": "hazard factor at 10 s", "name": term, "reference_term": REF[term],
                 "value": np.exp(10 * params["S5"]["M2'"][term]), "reference": np.exp(p12.loc[REF[term], "estimate"] * np.log(11))})

s8 = params["S8"]["M2'"]                  # S8: hazard factor at A = a vs A = 0, spline vs linear
for a in [0.25, 0.5, 0.75, 1.0]:
    x = np.r_[a, fit.rcs([a], KNOTS)[0]] - np.r_[0.0, fit.rcs([0.0], KNOTS)[0]]      # spline columns at a minus at 0
    rows.append({"sensitivity": "S8", "kind": "hazard factor vs A = 0", "name": f"A = {a}", "reference_term": "A",
                 "value": np.exp(x @ s8[["A", "A_rcs1", "A_rcs2"]].to_numpy()), "reference": np.exp(p12.loc["A", "estimate"] * a)})

weights = pd.DataFrame(rows)
weights["sign_same"] = weights["sign_same"].astype("boolean")
weights["inside_ci"] = weights["inside_ci"].astype("boolean")
weights["label"] = weights["sensitivity"].map(LABEL).fillna("")
weights = weights[["sensitivity", "kind", "name", "reference_term", "value", "se", "reference", "ref_ci_low", "ref_ci_high",
                   "sign_same", "inside_ci", "label"]]
weights.to_csv(config.P13_WEIGHTS, index=False)
weights[weights["kind"] == "M2' weight"].round(4)

,sensitivity,kind,name,reference_term,value,se,reference,ref_ci_low,ref_ci_high,sign_same,inside_ci,label
0,S1,M2' weight,const,const,-4.1077,0.0541,-3.1941,-3.3871,-3.0274,True,<NA>,
1,S1,M2' weight,lab,lab,0.0550,0.0267,0.0578,-0.1240,0.2136,True,True,
2,S1,M2' weight,A,A,-2.0931,0.0471,-2.1074,-2.4500,-1.7725,True,True,
3,S1,M2' weight,abs_s,abs_s,0.9834,0.0880,1.0476,0.8287,1.2760,True,True,
4,S1,M2' weight,at_end,at_end,0.4430,0.0611,0.4446,0.2966,0.5899,True,True,
5,S1,M2' weight,time_at_end_log1p,time_at_end_log1p,-0.3613,0.0239,-0.3774,-0.4901,-0.2792,True,True,
6,S1,M2' weight,time_since_rev_log1p,time_since_rev_log1p,-0.2006,0.0246,-0.2093,-0.3535,-0.0514,True,True,
7,S2,M2' weight,const,const,-2.5096,0.0529,-3.1941,-3.3871,-3.0274,True,<NA>,
8,S2,M2' weight,lab,lab,0.0529,0.0277,0.0578,-0.1240,0.2136,True,True,
9,S2,M2' weight,A,A,-2.1347,0.0488,-2.1074,-2.4500,-1.7725,True,True,


In [6]:
weights[weights["kind"] != "M2' weight"].round(4)

,sensitivity,kind,name,reference_term,value,se,reference,ref_ci_low,ref_ci_high,sign_same,inside_ci,label
57,S1,const shift vs ln(b/0.25),M1,NaN,-0.9166,NaN,-0.9163,NaN,NaN,<NA>,<NA>,
58,S1,const shift vs ln(b/0.25),M2,NaN,-0.9082,NaN,-0.9163,NaN,NaN,<NA>,<NA>,
59,S1,const shift vs ln(b/0.25),M2',NaN,-0.9135,NaN,-0.9163,NaN,NaN,<NA>,<NA>,
60,S2,const shift vs ln(b/0.25),M1,NaN,0.6847,NaN,0.6931,NaN,NaN,<NA>,<NA>,
61,S2,const shift vs ln(b/0.25),M2,NaN,0.6811,NaN,0.6931,NaN,NaN,<NA>,<NA>,
62,S2,const shift vs ln(b/0.25),M2',NaN,0.6845,NaN,0.6931,NaN,NaN,<NA>,<NA>,
63,S5,hazard factor at 10 s,time_at_end_cap10,time_at_end_log1p,0.3412,NaN,0.4045,NaN,NaN,<NA>,<NA>,
64,S5,hazard factor at 10 s,time_since_rev_cap10,time_since_rev_log1p,0.5496,NaN,0.6054,NaN,NaN,<NA>,<NA>,
65,S8,hazard factor vs A = 0,A = 0.25,A,0.5632,NaN,0.5905,NaN,NaN,<NA>,<NA>,descriptive
66,S8,hazard factor vs A = 0,A = 0.5,A,0.3803,NaN,0.3486,NaN,NaN,<NA>,<NA>,descriptive


In [7]:
print("M2' weights with a CI flag:", int(weights["inside_ci"].notna().sum()), "| inside the primary CI:", int(weights["inside_ci"].sum()))
print("M2' weights with a sign flag:", int(weights["sign_same"].notna().sum()), "| same sign:", int(weights["sign_same"].sum()))
weights[weights["kind"] == "M2' weight"].pivot(index="name", columns="sensitivity", values="inside_ci")

M2' weights with a CI flag: 43 | inside the primary CI: 41
M2' weights with a sign flag: 55 | same sign: 55


sensitivity,S1,S2,S3,S4,S5,S6,S7,S8
name,,,,,,,,
A,True,True,False,True,True,True,True,<NA>
A_rcs1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
A_rcs2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
abs_s,True,True,True,<NA>,True,True,True,True
at_end,True,True,False,<NA>,True,True,True,True
const,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
lab,True,True,True,True,True,True,True,True
time_at_end_cap10,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
time_at_end_log1p,True,True,True,True,<NA>,True,True,True
